# Data preparation

Download OpenR1-Math-220k, pick a clean trace per problem, extract
the boxed answer, filter long examples, deduplicate, split into
train/val/test, build a GRPO prompt-only version, and check for
GSM8K contamination before we train anything.

In [1]:
from datasets import load_dataset
import re, hashlib, json
from transformers import AutoTokenizer

TOKENIZER_NAME = 'Qwen/Qwen3-8B'
MAX_SEQ_LEN = 4096   # filter out longer samples for SFT
MAX_PROMPT_LEN = 512  # for GRPO prompts

In [2]:
# Load the 'default' split — about 94k problems where every problem
# has at least one correct reasoning trace. The 'extended' split adds
# more data but with worse quality, so we skip it.
ds = load_dataset('open-r1/OpenR1-Math-220k', 'default', split='train')
print(f'Raw size: {len(ds)}')
print(ds.column_names)
print(ds[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

data/train-00000-of-00010.parquet:   0%|          | 0.00/214M [00:00<?, ?B/s]

data/train-00001-of-00010.parquet:   0%|          | 0.00/215M [00:00<?, ?B/s]

...

data/train-00007-of-00010.parquet:   0%|          | 0.00/216M [00:00<?, ?B/s]

data/train-00008-of-00010.parquet:   0%|          | 0.00/214M [00:00<?, ?B/s]

data/train-00009-of-00010.parquet:   0%|          | 0.00/215M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/93733 [00:00<?, ? examples/s]

Raw size: 93733
['problem', 'solution', 'answer', 'problem_type', 'question_type', 'source', 'uuid', 'is_reasoning_complete', 'generations', 'correctness_math_verify', 'correctness_llama', 'finish_reasons', 'correctness_count', 'messages']
{'problem': '## Task B-1.3.\n\nA ship traveling along a river has covered $24 \\mathrm{~km}$ upstream and $28 \\mathrm{~km}$ downstream. For this journey, it took half an hour less than for traveling $30 \\mathrm{~km}$ upstream and $21 \\mathrm{~km}$ downstream, or half an hour more than for traveling $15 \\mathrm{~km}$ upstream and $42 \\mathrm{~km}$ downstream, assuming that both the ship and the river move uniformly.\n\nDetermine the speed of the ship in still water and the speed of the river.', 'solution': '## Solution.\n\nLet $t$ be the time required for the boat to travel $24 \\mathrm{~km}$ upstream and $28 \\mathrm{~km}$ downstream, $v_{R}$ the speed of the river, and $v_{B}$ the speed of the boat. When the boat is traveling upstream, its spee

In [3]:
# Each problem comes with multiple traces from DeepSeek-R1. Pick the
# first one that math-verify marked correct. If none are marked correct
# fall back to the first trace; this rarely happens in the default split.
def pick_best_trace(example):
    generations = example['generations']
    correctness = example['correctness_math_verify']
    for gen, correct in zip(generations, correctness):
        if correct:
            return {'chosen_trace': gen}
    return {'chosen_trace': generations[0]}  # fallback

ds = ds.map(pick_best_trace)
print(ds[0]['chosen_trace'][:300])

Map:   0%|          | 0/93733 [00:00<?, ? examples/s]

<think>
Okay, so I need to find the speed of the ship in still water and the speed of the river. Let me start by recalling that when a ship is moving upstream, its effective speed is the speed of the ship minus the speed of the river. Conversely, when moving downstream, its effective speed is the sh


In [4]:
# Pull the final answer out of the \boxed{} expression in the trace.
# If the regex misses, fall back to the dataset's 'answer' column.
# Drop anything we still can't extract an answer from.
def extract_boxed_answer(text):
    match = re.search(r'\\boxed\{([^}]*)\}', text)
    return match.group(1).strip() if match else None

ds = ds.map(lambda x: {'final_answer': extract_boxed_answer(x['chosen_trace']) or x.get('answer', '')})
# Drop rows with no extractable answer
ds = ds.filter(lambda x: x['final_answer'] != '')
print(f'After answer extraction: {len(ds)}')

Map:   0%|          | 0/93733 [00:00<?, ? examples/s]

Filter:   0%|          | 0/93733 [00:00<?, ? examples/s]

After answer extraction: 93733


In [5]:
# Build the SFT prompt in Qwen3's chat format. System prompt tells
# the model to box its final answer — this matters because our GRPO
# format reward later will check for the same \boxed{} marker.
# Keeping system, user, assistant in one templated string is what
# the SFTTrainer expects when packing is enabled.
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

SYSTEM_PROMPT = 'Please reason step by step, and put your final answer within \\boxed{}.'

def format_sft_example(example):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': example['problem']},
        {'role': 'assistant', 'content': example['chosen_trace']}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {'text': text}

ds = ds.map(format_sft_example)
print(ds[0]['text'][:500])

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Map:   0%|          | 0/93733 [00:00<?, ? examples/s]

<|im_start|>system
Please reason step by step, and put your final answer within \boxed{}.<|im_end|>
<|im_start|>user
## Task B-1.3.

A ship traveling along a river has covered $24 \mathrm{~km}$ upstream and $28 \mathrm{~km}$ downstream. For this journey, it took half an hour less than for traveling $30 \mathrm{~km}$ upstream and $21 \mathrm{~km}$ downstream, or half an hour more than for traveling $15 \mathrm{~km}$ upstream and $42 \mathrm{~km}$ downstream, assuming that both the ship and the ri


In [6]:
# Filter out anything longer than MAX_SEQ_LEN tokens. Some olympiad
# problems have very long solutions and they don't fit on A100 40GB.
# Losing a few percent of the data is worth keeping training stable.
def token_length(example):
    return {'n_tokens': len(tokenizer(example['text'], truncation=False)['input_ids'])}

ds = ds.map(token_length, num_proc=4)
ds_sft = ds.filter(lambda x: x['n_tokens'] <= MAX_SEQ_LEN)
print(f'After length filter (<={MAX_SEQ_LEN} tokens): {len(ds_sft)}')

Map (num_proc=4):   0%|          | 0/93733 [00:00<?, ? examples/s]

Filter:   0%|          | 0/93733 [00:00<?, ? examples/s]

After length filter (<=4096 tokens): 37828


In [7]:
# Quick dedup by hashing the problem statement. Same problem can show
# up multiple times across NuminaMath sources (e.g. cn_contest and
# aops_forum overlap a lot).
seen = set()
def is_unique(example):
    h = hashlib.md5(example['problem'].strip().lower().encode()).hexdigest()
    if h in seen:
        return False
    seen.add(h)
    return True

ds_sft = ds_sft.filter(is_unique)
print(f'After dedup: {len(ds_sft)}')

Filter:   0%|          | 0/37828 [00:00<?, ? examples/s]

After dedup: 37822


In [8]:
# Standard 80/10/10 split with a fixed seed. The test set is going
# to be frozen — we never look at it during training and never use
# it to pick adapters or hyperparameters.
ds_split = ds_sft.train_test_split(test_size=0.2, seed=42)
ds_val_test = ds_split['test'].train_test_split(test_size=0.5, seed=42)

train_ds = ds_split['train']
val_ds   = ds_val_test['train']
test_ds  = ds_val_test['test']

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

Train: 30257 | Val: 3782 | Test: 3783


In [9]:
# Build a smaller dataset of problem + gold answer for GRPO.
# GRPO doesn't need the reasoning trace, just the prompt and a
# verifiable answer. Filter prompts to MAX_PROMPT_LEN tokens so
# rollout generation has room for completions.
grpo_ds = train_ds.select_columns(['problem', 'final_answer'])
grpo_ds = grpo_ds.map(lambda x: {
    'prompt': [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': x['problem']}
    ]
})
# Filter prompts to MAX_PROMPT_LEN tokens
grpo_ds = grpo_ds.filter(
    lambda x: len(tokenizer.apply_chat_template(x['prompt'], tokenize=True)) <= MAX_PROMPT_LEN
)
print(f'GRPO prompt dataset: {len(grpo_ds)}')

Map:   0%|          | 0/30257 [00:00<?, ? examples/s]

Filter:   0%|          | 0/30257 [00:00<?, ? examples/s]

GRPO prompt dataset: 30257


In [10]:
# Contamination check. Hash every GSM8K test problem and intersect
# with our training problems. Expect zero overlap; if it's nonzero
# we'd have to drop those rows before training.
gsm8k_test = load_dataset('gsm8k', 'main', split='test')
gsm8k_hashes = set(hashlib.md5(p.strip().lower().encode()).hexdigest() for p in gsm8k_test['question'])
train_hashes = set(hashlib.md5(p.strip().lower().encode()).hexdigest() for p in train_ds['problem'])

overlap = gsm8k_hashes & train_hashes
print(f'GSM8K test problems in our train set: {len(overlap)} (should be 0)')

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

GSM8K test problems in our train set: 0 (should be 0)


In [11]:
# Save everything to /content/data. After this cell, immediately
# copy these folders to Google Drive so we don't lose them when
# Colab disconnects.
train_ds.save_to_disk('/content/data/sft_train')
val_ds.save_to_disk('/content/data/sft_val')
test_ds.save_to_disk('/content/data/sft_test')
grpo_ds.save_to_disk('/content/data/grpo_prompts')
print('Datasets saved.')

Saving the dataset (0/6 shards):   0%|          | 0/30257 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3782 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3783 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/30257 [00:00<?, ? examples/s]

Datasets saved.


In [12]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
os.makedirs('/content/drive/MyDrive/llm_posttraining/data', exist_ok=True)

shutil.copytree('/content/data/sft_train',    '/content/drive/MyDrive/llm_posttraining/data/sft_train',    dirs_exist_ok=True)
shutil.copytree('/content/data/sft_val',      '/content/drive/MyDrive/llm_posttraining/data/sft_val',      dirs_exist_ok=True)
shutil.copytree('/content/data/sft_test',     '/content/drive/MyDrive/llm_posttraining/data/sft_test',     dirs_exist_ok=True)
shutil.copytree('/content/data/grpo_prompts', '/content/drive/MyDrive/llm_posttraining/data/grpo_prompts', dirs_exist_ok=True)
print('Data backed up to Drive.')

Mounted at /content/drive
Data backed up to Drive.
